In [ ]:
import json
import numpy as np
import pandas as pd
import nibabel as nib
from nibabel.affines import apply_affine
from skimage.morphology import skeletonize
from skimage.morphology import thin
nii    = nib.load("/Users/jlepinay/SPIMED/DataStorage/ImageCAS_Batch1_Stack0/scan.nii")
data   = nii.get_fdata()
niiCoro = nib.load("/Users/jlepinay/SPIMED/DataStorage/ImageCAS_Batch1_Stack0/coroBinMask01.nii")
dataCoro = niiCoro.get_fdata()
print(np.unique(dataCoro))
print(dataCoro.shape)
print(np.count_nonzero(dataCoro))

skel = skeletonize(dataCoro)
print(skel.shape)
print("Skeleton unique values:", np.unique(skel))
print("Skeleton non-zero count:", np.count_nonzero(skel))

In [ ]:
import SkelToGraph

In [ ]:
from scipy.sparse import csr_matrix
from scipy.ndimage import generate_binary_structure, label, find_objects
from scipy.spatial import cKDTree

def skeleton_to_graph(skel):
    # Label connected components
    struct = generate_binary_structure(3, 3)  # 26-neighborhood
    labeled, num = label(skel, structure=struct)
    points = np.array(np.where(skel)).T

    # For each point, find its neighbors (max 26)
    # Build adjacency dictionary
    adj = {}
    for idx, p in enumerate(points):
        adj[idx] = set()
    # Use a KDTree for fast neighbor search
    tree = cKDTree(points)
    # Query all points within distance sqrt(3) (since max step is 1 in 26-neighborhood)
    for i, p in enumerate(points):
        # find points within distance 1.1 (accounts for diagonal neighbors)
        neighbors = tree.query_ball_point(p, r=1.1)
        for j in neighbors:
            if i != j:
                d = np.linalg.norm(points[i] - points[j])
                if d < 1.2:  # only if truly adjacent in skeleton
                    adj[i].add(j)
    # Build adjacency matrix / sparse graph
    n = len(points)
    rows, cols, data = [], [], []
    for i in adj:
        for j in adj[i]:
            if i < j:
                d = np.linalg.norm(points[i] - points[j])
                rows.append(i); cols.append(j); data.append(d)
                rows.append(j); cols.append(i); data.append(d)
    graph = csr_matrix((data, (rows, cols)), shape=(n, n))
    return graph, points

In [ ]:
skel_graph, skel_points = skeleton_to_graph(skel)
print("Graph shape:", skel_graph.shape)

In [ ]:
SkelToGraph.plot_graph_3d(skel_graph, skel_points, output_file='graph_3d.html', node_size=3, edge_width=4, opacity=0.8, node_color='blue', edge_color='red', title='3D Graph Visualization')

In [ ]:
skel_graph, skel_points, _,_ = SkelToGraph.Skel2Graph(skel, 26)
SkelToGraph.plot_graph_3d(skel_graph, skel_points, output_file='graph_3d.html', node_size=3, edge_width=4, opacity=0.8, node_color='blue', edge_color='red', title='3D Graph Visualization')

In [ ]:
skel_graph, skel_points, _,_ = SkelToGraph.Skel2Graph(skel, 6)
SkelToGraph.plot_graph_3d(skel_graph, skel_points, output_file='graph_3d.html', node_size=3, edge_width=4, opacity=0.8, node_color='blue', edge_color='red', title='3D Graph Visualization')

In [ ]:
SkelToGraph.draw_graph_graphviz(skel_graph, points=None, layout='neato', output_file='graph.gv.svg',
                        node_color='blue', edge_color='gray', prog='neato', dpi=1000)

In [ ]:
from collections import defaultdict
import numpy as np
import pandas as pd
import networkx as nx
from scipy.sparse import csr_matrix

def get_cycle_length(G: nx.Graph, cycle: list[int]) -> float | None:
    total = 0.0
    n = len(cycle)
    for i in range(n):
        u, v = cycle[i], cycle[(i + 1) % n]
        if G.has_edge(u, v):
            total += float(G[u][v].get("weight", 1.0))
        elif G.has_edge(v, u):
            total += float(G[v][u].get("weight", 1.0))
        else:
            return None
    return total

def _cycle_edges(cycle: list[int]) -> set[tuple[int, int]]:
    return {
        tuple(sorted((cycle[i], cycle[(i + 1) % len(cycle)])))
        for i in range(len(cycle))
    }

def greedy_merge_cycles(
    G: nx.Graph,
    min_cycle_edges: int = 4,
    min_shared_nodes: int = 2,
    keep_triangles_touching_large: bool = True,
    triangle_touch_nodes: int = 2,
    verbose: bool = True,
) -> dict:
    """
    Greedy grouping of basis cycles.

    Rules:
    - cycles with size >= min_cycle_edges are candidates
    - triangles (size 3) are skipped unless they share >= triangle_touch_nodes
      nodes with a large cycle
    - cycles are merged into the first/best group they overlap with by
      at least min_shared_nodes nodes
    """

    raw_cycles = nx.cycle_basis(G)

    records = []
    for cid, cyc in enumerate(raw_cycles):
        cyc = [int(n) for n in cyc]
        rec = {
            "cycle_id": cid,
            "cycle": cyc,
            "size": len(cyc),
            "length": get_cycle_length(G, cyc),
            "nodes": set(cyc),
            "edges": _cycle_edges(cyc),
        }
        records.append(rec)

    large_node_sets = [r["nodes"] for r in records if r["size"] >= min_cycle_edges]

    accepted = []
    for r in records:
        if r["size"] >= min_cycle_edges:
            accepted.append(r)
        elif r["size"] == 3 and keep_triangles_touching_large:
            touches_large = any(
                len(r["nodes"] & s) >= triangle_touch_nodes
                for s in large_node_sets
            )
            if touches_large:
                accepted.append(r)

    # Process large cycles first so they become group seeds.
    accepted.sort(key=lambda r: (r["size"], r["length"]), reverse=True)

    groups = []
    cycle_to_group = {}

    for r in accepted:
        best_gid = None
        best_overlap = 0

        for gid, grp in enumerate(groups):
            overlap = len(r["nodes"] & grp["nodes"])
            if overlap > best_overlap:
                best_overlap = overlap
                best_gid = gid

        if best_gid is not None and best_overlap >= min_shared_nodes:
            grp = groups[best_gid]
            grp["cycles"].append(r)
            grp["nodes"].update(r["nodes"])
            grp["edges"].update(r["edges"])
            grp["lengths"].append(r["length"])
            cycle_to_group[r["cycle_id"]] = best_gid
        else:
            gid = len(groups)
            groups.append({
                "group_id": gid,
                "cycles": [r],
                "nodes": set(r["nodes"]),
                "edges": set(r["edges"]),
                "lengths": [r["length"]],
            })
            cycle_to_group[r["cycle_id"]] = gid

    # Decorate group summaries
    for grp in groups:
        grp["n_cycles"] = len(grp["cycles"])
        grp["n_nodes"] = len(grp["nodes"])
        grp["n_edges"] = len(grp["edges"])
        grp["total_cycle_length"] = float(np.nansum(grp["lengths"]))
        grp["max_cycle_length"] = float(np.nanmax(grp["lengths"])) if grp["lengths"] else 0.0
        grp["representative_cycle"] = max(
            grp["cycles"],
            key=lambda r: (r["size"], r["length"])
        )["cycle"]
        grp["cycle_ids"] = [r["cycle_id"] for r in grp["cycles"]]
        grp["cycle_sizes"] = [r["size"] for r in grp["cycles"]]

    # Per-node and per-edge membership
    node_to_cycles = defaultdict(list)
    node_to_groups = defaultdict(list)
    edge_to_cycles = defaultdict(list)
    edge_to_groups = defaultdict(list)

    for r in records:
        gid = cycle_to_group.get(r["cycle_id"], -1)
        for n in r["nodes"]:
            node_to_cycles[n].append(r["cycle_id"])
        for e in r["edges"]:
            edge_to_cycles[e].append(r["cycle_id"])
        if gid != -1:
            for n in r["nodes"]:
                node_to_groups[n].append(gid)
            for e in r["edges"]:
                edge_to_groups[e].append(gid)

    # Tables
    cycle_rows = []
    for r in records:
        gid = cycle_to_group.get(r["cycle_id"], -1)
        cycle_rows.append({
            "cycle_id": r["cycle_id"],
            "group_id": gid,
            "accepted": gid != -1,
            "size": r["size"],
            "length": r["length"],
            "nodes": sorted(r["nodes"]),
            "edges": sorted(r["edges"]),
        })
    cycle_df = pd.DataFrame(cycle_rows)

    node_rows = []
    for n in G.nodes():
        node_rows.append({
            "node": int(n),
            "cycle_count": len(node_to_cycles.get(n, [])),
            "group_count": len(node_to_groups.get(n, [])),
            "cycles": sorted(set(node_to_cycles.get(n, []))),
            "groups": sorted(set(node_to_groups.get(n, []))),
        })
    node_df = pd.DataFrame(node_rows)

    all_edges = sorted({tuple(sorted((u, v))) for u, v in G.edges()})
    edge_rows = []
    for e in all_edges:
        edge_rows.append({
            "u": e[0],
            "v": e[1],
            "cycle_count": len(edge_to_cycles.get(e, [])),
            "group_count": len(edge_to_groups.get(e, [])),
            "cycles": sorted(set(edge_to_cycles.get(e, []))),
            "groups": sorted(set(edge_to_groups.get(e, []))),
        })
    edge_df = pd.DataFrame(edge_rows)

    group_rows = []
    for grp in groups:
        group_rows.append({
            "group_id": grp["group_id"],
            "n_cycles": grp["n_cycles"],
            "n_nodes": grp["n_nodes"],
            "n_edges": grp["n_edges"],
            "total_cycle_length": grp["total_cycle_length"],
            "max_cycle_length": grp["max_cycle_length"],
            "cycle_ids": grp["cycle_ids"],
            "cycle_sizes": grp["cycle_sizes"],
            "representative_cycle": grp["representative_cycle"],
        })
    group_df = pd.DataFrame(group_rows)

    if verbose:
        print(f"Raw basis cycles: {len(records)}")
        print(f"Accepted cycles: {len(accepted)}")
        print(f"Merge groups: {len(groups)}")
        print("Group sizes:", group_df["n_cycles"].tolist() if len(group_df) else [])

    return {
        "raw_cycles": records,
        "accepted_cycles": accepted,
        "groups": groups,
        "cycle_df": cycle_df,
        "node_df": node_df,
        "edge_df": edge_df,
        "group_df": group_df,
    }

In [ ]:
skel_graph, skel_points, _, _ = SkelToGraph.Skel2Graph(skel, 6)

G = SkelToGraph.CSR2NetworkX(skel_graph, skel_points)

cycle_info = greedy_merge_cycles(
    G,
    min_cycle_edges=4,              # skip triangles by default
    min_shared_nodes=2,             # merge only real overlaps
    keep_triangles_touching_large=True,
    triangle_touch_nodes=2,         # keep triangle only if it shares 2 nodes with a large cycle
    verbose=True,
)

print(cycle_info["group_df"])
print(cycle_info["cycle_df"].head())
print(cycle_info["node_df"].head())
print(cycle_info["edge_df"].head())

SkelToGraph.draw_graph_graphviz_cycles(
    G,
    #points=skel_points[:, :2],      # Graphviz is 2D, so use x/y
    cycle_groups=cycle_info["groups"],
    output_file="graph_cycles.svg",
    prog="neato",
)

In [ ]:
import json
import numpy as np
import pandas as pd
import nibabel as nib
from nibabel.affines import apply_affine
from skimage.morphology import skeletonize
from skimage.morphology import thin
nii    = nib.load("/Users/jlepinay/SPIMED/DataStorage/ImageCAS_Batch1_Stack0/scan.nii")
data   = nii.get_fdata()
niiCoro = nib.load("/Users/jlepinay/SPIMED/DataStorage/ImageCAS_Batch1_Stack0/coroBinMask01.nii")
dataCoro = niiCoro.get_fdata()
print(np.unique(dataCoro))
print(dataCoro.shape)
print(np.count_nonzero(dataCoro))

skel = skeletonize(dataCoro)
print(skel.shape)
print("Skeleton unique values:", np.unique(skel))
print("Skeleton non-zero count:", np.count_nonzero(skel))

import SkelToGraph

# --- skeleton -> graph ---
graph, points, n_components, labels = SkelToGraph.Skel2Graph(skel, k_neighbors=6)
print("Components:", n_components)

# --- graph -> NetworkX ---
G = SkelToGraph.CSR2NetworkX(graph, points)

# --- extract cycles and group them ---
cycle_result = SkelToGraph.process_cycles(
    G,
    min_cycle_edges=4,
    min_shared_nodes=2,
    keep_triangles_touching_large=True,
    triangle_touch_nodes=2
)

print(cycle_result["cycle_df"])
print(cycle_result["group_df"])

# --- prepare cycle_groups for your plotting function ---
cycle_groups = []
for g in cycle_result["groups"]:
    all_edges = set()
    for cyc in g["cycles"]:
        all_edges |= cyc["edges"]

    cycle_groups.append({
        "group_id": g["group_id"],
        "edges": list(all_edges),
        "nodes": g["nodes"],
    })

# --- plot cycles with Graphviz ---
SkelToGraph.draw_graph_graphviz_cycles(
    graph=graph,
    cycle_groups=cycle_groups,
    output_file="cycles.svg",
    base_edge_width=1,
    cycle_edge_width=3,
    multi_cycle_edge_width=5
)

# --- optional 3D interactive plot ---
SkelToGraph.plot_graph_3d(graph, points, output_file="graph_3d.html")

In [ ]:
import networkx as nx
import numpy as np
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.sparse import csr_matrix
from collections import defaultdict


# ─────────────────────────────────────────────
# 1.  PARTITION
# ─────────────────────────────────────────────

def partition_graph(G: nx.Graph, cycle_results: dict) -> tuple[set[int], set[int]]:
    """
    Revised — nodes that appear in cycle groups but also connect to
    non-cycle nodes are demoted to clean_nodes (they become junctions).
    Pure cycle-interior nodes (all neighbours are also in cycles) stay
    in cycle_nodes.
    """
    raw_cycle_nodes: set[int] = set()
    for grp in cycle_results["groups"]:
        raw_cycle_nodes.update(grp["nodes"])

    # Promote boundary cycle nodes back to clean so they act as junctions
    cycle_nodes  = set()
    clean_nodes  = set(G.nodes()) - raw_cycle_nodes

    for n in raw_cycle_nodes:
        neighbours = set(G.neighbors(n))
        if neighbours - raw_cycle_nodes:
            # Has at least one neighbour outside the cycle region → boundary
            clean_nodes.add(n)
            G.nodes[n]["junction"] = True
        else:
            cycle_nodes.add(n)
            G.nodes[n]["junction"] = False

    return cycle_nodes, clean_nodes


def extract_subgraphs(G: nx.Graph,
                      cycle_nodes: set[int],
                      clean_nodes: set[int]) -> tuple[nx.Graph, nx.Graph, set[int]]:
    """
    Fixed version — returns junction nodes as a third value so callers
    can inspect them.
    """
    # Junction = clean node that shares at least one edge with a cycle node
    junction_nodes = set()
    for u, v in G.edges():
        if u in cycle_nodes and v in clean_nodes:
            junction_nodes.add(v)
        elif v in cycle_nodes and u in clean_nodes:
            junction_nodes.add(u)

    # Tag in the original graph so subgraphs inherit the attribute
    for n in G.nodes():
        G.nodes[n]["junction"] = (n in junction_nodes)

    clean_graph = G.subgraph(clean_nodes | junction_nodes).copy()
    cycle_graph = G.subgraph(cycle_nodes | junction_nodes).copy()

    return clean_graph, cycle_graph, junction_nodes


# ─────────────────────────────────────────────
# 2a.  MST ON CLEAN SUBGRAPH
# ─────────────────────────────────────────────

def mst_subgraph(G: nx.Graph) -> nx.Graph:
    """Minimum spanning tree of a weighted NetworkX graph."""
    nodes  = list(G.nodes())
    idx    = {n: i for i, n in enumerate(nodes)}
    n      = len(nodes)
    mat    = np.zeros((n, n))
    for u, v, d in G.edges(data=True):
        w = d.get("weight", 1.0)
        mat[idx[u], idx[v]] = w
        mat[idx[v], idx[u]] = w
    mst = minimum_spanning_tree(csr_matrix(mat))
    mst_G = nx.Graph()
    mst_G.add_nodes_from([(n, G.nodes[n]) for n in nodes])
    cx = mst.tocoo()
    for i, j, w in zip(cx.row, cx.col, cx.data):
        if i < j:
            mst_G.add_edge(nodes[i], nodes[j], weight=float(w))
    return mst_G


# ─────────────────────────────────────────────
# 2b.  PRUNE EACH CYCLE GROUP
# ─────────────────────────────────────────────

def _steiner_star(G: nx.Graph, terminals: list[int]) -> nx.Graph:
    """
    Approximate Steiner tree connecting `terminals` inside G.
    Strategy: shortest-path tree from the terminal closest to all others
    (metric centre approximation – good enough for small, noisy loops).
    """
    # pick the terminal with smallest total distance to every other terminal
    best_root, best_total = terminals[0], float("inf")
    for t in terminals:
        try:
            total = sum(
                nx.shortest_path_length(G, t, other, weight="weight")
                for other in terminals if other != t
            )
            if total < best_total:
                best_root, best_total = t, total
        except nx.NetworkXNoPath:
            pass

    star = nx.Graph()
    star.add_nodes_from([(n, G.nodes[n]) for n in G.nodes()])
    for t in terminals:
        if t == best_root:
            continue
        try:
            path = nx.shortest_path(G, best_root, t, weight="weight")
            for a, b in zip(path, path[1:]):
                star.add_edge(a, b, **G[a][b])
        except nx.NetworkXNoPath:
            pass
    return star


def prune_cycle_group(group: dict, cycle_subgraph: nx.Graph) -> nx.Graph:
    """
    Replace a single cycle group with a minimal path or bifurcation.

    Rules
    -----
    * Collect the *junction nodes* inside this group (nodes tagged 'junction').
    * 0 or 1 junction  → isolated noise; return empty graph (drop entirely).
    * 2 junctions      → straight path (shortest path between them).
    * 3+ junctions     → Steiner-star approximation connecting all junctions.
    """
    group_nodes = group["nodes"]
    sub = cycle_subgraph.subgraph(group_nodes).copy()

    junction_nodes = [
        n for n in group_nodes
        if cycle_subgraph.nodes[n].get("junction", False)
    ]

    pruned = nx.Graph()
    pruned.add_nodes_from([(n, cycle_subgraph.nodes[n]) for n in group_nodes])

    if len(junction_nodes) < 2:
        # isolated noise – drop
        return pruned

    if len(junction_nodes) == 2:
        u, v = junction_nodes
        try:
            path = nx.shortest_path(sub, u, v, weight="weight")
            for a, b in zip(path, path[1:]):
                pruned.add_edge(a, b, **sub[a][b])
        except nx.NetworkXNoPath:
            # fallback: direct edge with Euclidean weight if positions exist
            pos_u = cycle_subgraph.nodes[u].get("pos")
            pos_v = cycle_subgraph.nodes[v].get("pos")
            w = float(np.linalg.norm(pos_u - pos_v)) if (pos_u is not None and pos_v is not None) else 1.0
            pruned.add_edge(u, v, weight=w)
        return pruned

    # 3+ junctions → star approximation
    return _steiner_star(sub, junction_nodes)


def prune_all_cycle_groups(cycle_results: dict,
                           cycle_subgraph: nx.Graph) -> list[nx.Graph]:
    """Return one pruned subgraph per cycle group."""
    return [
        prune_cycle_group(grp, cycle_subgraph)
        for grp in cycle_results["groups"]
    ]


# ─────────────────────────────────────────────
# 3.  RECOMBINE
# ─────────────────────────────────────────────

def recombine(mst_clean: nx.Graph,
              pruned_cycle_graphs: list[nx.Graph],
              original_G: nx.Graph) -> nx.Graph:
    """
    Merge the MST of the clean subgraph with all pruned cycle replacements.

    Only edges that appear in at least one of the component graphs are kept.
    Junction nodes are already shared, so no extra stitching is needed.
    """
    final = nx.Graph()

    # Start from all nodes in the original graph (preserving attributes)
    for n, attrs in original_G.nodes(data=True):
        final.add_node(n, **attrs)

    # Add MST edges
    for u, v, d in mst_clean.edges(data=True):
        final.add_edge(u, v, **d)

    # Add pruned cycle edges
    for pg in pruned_cycle_graphs:
        for u, v, d in pg.edges(data=True):
            if not final.has_edge(u, v):
                final.add_edge(u, v, **d)

    return final


# ─────────────────────────────────────────────
# TOP-LEVEL CONVENIENCE FUNCTION
# ─────────────────────────────────────────────

def simplify_graph(G: nx.Graph,
                   cycle_results: dict,
                   verbose: bool = True) -> nx.Graph:

    # 1. Partition — boundary cycle nodes become junctions in clean_nodes
    cycle_nodes, clean_nodes = partition_graph(G, cycle_results)

    if verbose:
        junction_count = sum(1 for n in clean_nodes if G.nodes[n].get("junction"))
        print(f"Cycle interior nodes : {len(cycle_nodes)}")
        print(f"Clean nodes          : {len(clean_nodes)}  ({junction_count} junctions)")

    clean_sub, cycle_sub, junction_nodes = extract_subgraphs(G, cycle_nodes, clean_nodes)

    if verbose:
        print(f"Cycle subgraph connected components : "
              f"{nx.number_connected_components(cycle_sub)}")
        print(f"Clean subgraph connected components : "
              f"{nx.number_connected_components(clean_sub)}")

    # 2a. MST on clean part
    mst_clean = mst_subgraph(clean_sub)

    # 2b. Prune each cycle group
    pruned_graphs = prune_all_cycle_groups(cycle_results, cycle_sub)

    if verbose:
        pruned_edges = sum(pg.number_of_edges() for pg in pruned_graphs)
        print(f"Pruned cycle groups : {len(pruned_graphs)}")
        print(f"Pruned cycle edges  : {pruned_edges}")

    # 3. Recombine
    final_G = recombine(mst_clean, pruned_graphs, G)

    if verbose:
        print(f"\nOriginal edges : {G.number_of_edges()}")
        print(f"Final    edges : {final_G.number_of_edges()}")
        reduction = (1 - final_G.number_of_edges() / G.number_of_edges()) * 100
        print(f"Reduction      : {reduction:.1f}%")

    return final_G

def debug_partition(G: nx.Graph, cycle_results: dict):
    cycle_nodes, clean_nodes = partition_graph(G, cycle_results)
    clean_sub, cycle_sub, junctions = extract_subgraphs(G, cycle_nodes, clean_nodes)

    print(f"Junction nodes: {sorted(junctions)[:20]} ...")
    print(f"Cycle subgraph: {cycle_sub.number_of_nodes()} nodes, "
          f"{cycle_sub.number_of_edges()} edges, "
          f"{nx.number_connected_components(cycle_sub)} components")
    print(f"Clean subgraph: {clean_sub.number_of_nodes()} nodes, "
          f"{clean_sub.number_of_edges()} edges, "
          f"{nx.number_connected_components(clean_sub)} components")

    # Each cycle group should now be connected within cycle_sub
    for i, grp in enumerate(cycle_results["groups"]):
        grp_nodes = grp["nodes"] | junctions  # include junctions
        sub = cycle_sub.subgraph(grp_nodes & set(cycle_sub.nodes()))
        cc = nx.number_connected_components(sub)
        if cc > 1:
            print(f"  WARNING group {i}: still {cc} components")
        else:
            print(f"  Group {i}: OK ({len(grp_nodes)} nodes)")

In [ ]:
graph_csr, points, n_components, labels = SkelToGraph.Skel2Graph(skel)
G = SkelToGraph.CSR2NetworkX(graph_csr, points)

# Detect and group cycles (your existing function)
cycle_results = SkelToGraph.process_cycles(G, min_cycle_edges=4, min_shared_nodes=2)

# Simplify
G_simplified = simplify_graph(G, cycle_results, verbose=True)

# Visualise both for comparison
#SkelToGraph.plot_graph_3d(G,            points, output_file='original.html',   title='Original')
SkelToGraph.plot_graph_3d(G_simplified, points, output_file='simplified.html', title='Simplified')


In [ ]:
SkelToGraph.plot_graph_3d(G,            points, output_file='original.html',   title='Original')

In [ ]:
print(f"Original graph: {nx.number_connected_components(G)} components")
print(f"Simplified graph: {nx.number_connected_components(G_simplified)} components")

In [ ]:
isolated_nodes = [node for node in G.nodes() if G_simplified.degree(node) == 0]
len(isolated_nodes)

In [ ]:
from scipy.spatial import cKDTree
import numpy as np
import networkx as nx
from scipy.sparse.csgraph import connected_components
from scipy.sparse import csr_matrix


def get_component_labels(G: nx.Graph) -> dict[int, int]:
    """Map node → component id using networkx (handles non-integer node ids)."""
    return {n: i for i, comp in enumerate(nx.connected_components(G)) 
            for n in comp}


def bridge_components(G: nx.Graph,
                      points: np.ndarray,
                      target_components: int = 1,
                      max_bridge_distance: float = None,
                      verbose: bool = True) -> nx.Graph:
    """
    Iteratively connect the closest pair of nodes across component boundaries
    until we reach `target_components` connected components (or no more
    bridges are possible within `max_bridge_distance`).

    Parameters
    ----------
    G                  : NetworkX graph (nodes are integer indices into points)
    points             : (N, 3) array of 3D coordinates
    target_components  : desired number of connected components (default 1)
    max_bridge_distance: maximum Euclidean distance allowed for a bridge edge.
                         None = no limit (always bridge).
    verbose            : print progress

    Returns
    -------
    G_connected : copy of G with bridge edges added
    """
    G = G.copy()
    tree = cKDTree(points)

    iteration = 0
    while True:
        # Current component assignment
        comp_labels = get_component_labels(G)
        n_comp = len(set(comp_labels.values()))

        if verbose:
            print(f"Iteration {iteration}: {n_comp} component(s)")

        if n_comp <= target_components:
            break

        # For each component, find its closest node in any OTHER component
        # Strategy: for every node, query its nearest neighbours until we
        # find one in a different component
        best_dist = np.inf
        best_pair = None

        # Group nodes by component
        comp_to_nodes: dict[int, list[int]] = {}
        for node, cid in comp_labels.items():
            comp_to_nodes.setdefault(cid, []).append(node)

        # For efficiency: query each component's points against the full tree,
        # find the nearest point that belongs to a different component
        for cid, nodes in comp_to_nodes.items():
            comp_points = points[nodes]
            # k=2 because k=1 might return itself; use larger k to cross components
            k = min(len(points), 20)
            dists, idxs = tree.query(comp_points, k=k)

            for local_i, (row_dists, row_idxs) in enumerate(zip(dists, idxs)):
                src_node = nodes[local_i]
                for d, tgt_node in zip(row_dists, row_idxs):
                    if comp_labels[tgt_node] != cid:
                        if d < best_dist:
                            best_dist = d
                            best_pair = (src_node, tgt_node)
                        break  # nearest cross-component neighbour found for this node

        if best_pair is None:
            if verbose:
                print("No cross-component pairs found. Stopping.")
            break

        if max_bridge_distance is not None and best_dist > max_bridge_distance:
            if verbose:
                print(f"Closest bridge is {best_dist:.2f} > max_bridge_distance "
                      f"({max_bridge_distance:.2f}). Stopping.")
            break

        u, v = best_pair
        G.add_edge(u, v, weight=float(best_dist), bridge=True)

        if verbose:
            print(f"  → Bridge added: node {u} ↔ node {v}  (dist={best_dist:.3f})")

        iteration += 1

    final_comp = len(set(get_component_labels(G).values()))
    if verbose:
        print(f"\nDone. Final component count: {final_comp}")

    return G


def bridge_components_batch(G: nx.Graph,
                             points: np.ndarray,
                             target_components: int = 1,
                             max_bridge_distance: float = None,
                             verbose: bool = True) -> nx.Graph:
    """
    Faster version for many components: in each iteration, connect ALL
    component pairs simultaneously (one bridge per pair of adjacent components
    in the minimum spanning tree of the component-distance graph).

    Better when you have dozens of small disconnected fragments.
    """
    G = G.copy()
    tree = cKDTree(points)

    iteration = 0
    while True:
        comp_labels = get_component_labels(G)
        comp_ids = list(set(comp_labels.values()))
        n_comp = len(comp_ids)

        if verbose:
            print(f"Iteration {iteration}: {n_comp} component(s)")

        if n_comp <= target_components:
            break

        # Build a complete graph of inter-component distances
        # node = component id, edge weight = min distance between any two nodes
        comp_to_nodes = {}
        for node, cid in comp_labels.items():
            comp_to_nodes.setdefault(cid, []).append(node)

        # For each component, find closest node in every other component
        # Use a dict: (cid_a, cid_b) → (dist, node_a, node_b)
        inter: dict[tuple, tuple] = {}

        for cid, nodes in comp_to_nodes.items():
            comp_points = points[nodes]
            k = min(len(points), 50)
            dists, idxs = tree.query(comp_points, k=k)

            for local_i, (row_dists, row_idxs) in enumerate(zip(dists, idxs)):
                src_node = nodes[local_i]
                seen_comps = set()
                for d, tgt_node in zip(row_dists, row_idxs):
                    tgt_cid = comp_labels[tgt_node]
                    if tgt_cid == cid or tgt_cid in seen_comps:
                        continue
                    seen_comps.add(tgt_cid)
                    key = (min(cid, tgt_cid), max(cid, tgt_cid))
                    if key not in inter or d < inter[key][0]:
                        inter[key] = (d, src_node, tgt_node)

        if not inter:
            break

        # MST of the component graph → minimum set of bridges to connect all
        comp_idx = {cid: i for i, cid in enumerate(comp_ids)}
        n = len(comp_ids)
        mat = np.full((n, n), np.inf)
        np.fill_diagonal(mat, 0)
        for (ca, cb), (d, _, _) in inter.items():
            i, j = comp_idx[ca], comp_idx[cb]
            mat[i, j] = mat[j, i] = d

        from scipy.sparse.csgraph import minimum_spanning_tree
        from scipy.sparse import csr_matrix
        # Replace inf with 0 for CSR (zeros = no edge, so use finite values only)
        finite_mat = np.where(np.isinf(mat), 0, mat)
        mst_bridges = minimum_spanning_tree(csr_matrix(finite_mat)).tocoo()

        bridges_added = 0
        for i, j, d in zip(mst_bridges.row, mst_bridges.col, mst_bridges.data):
            if i >= j or d == 0:
                continue
            ca = comp_ids[i]
            cb = comp_ids[j]
            key = (min(ca, cb), max(ca, cb))
            dist, u, v = inter[key]

            if max_bridge_distance is not None and dist > max_bridge_distance:
                if verbose:
                    print(f"  Skipping bridge {u}↔{v}: dist={dist:.2f} > limit")
                continue

            G.add_edge(u, v, weight=float(dist), bridge=True)
            bridges_added += 1
            if verbose:
                print(f"  Bridge: node {u} ↔ node {v}  (dist={dist:.3f})")

        if bridges_added == 0:
            if verbose:
                print("No valid bridges added. Stopping.")
            break

        iteration += 1

    final_comp = len(set(get_component_labels(G).values()))
    if verbose:
        print(f"\nDone. Final component count: {final_comp}")

    return G

In [ ]:
# After Skel2Graph
graph_csr, points, n_components, labels = SkelToGraph.Skel2Graph(skel, k_neighbors=6)
G = SkelToGraph.CSR2NetworkX(graph_csr, points)

print(f"Components before bridging: {nx.number_connected_components(G)}")

# If you have a few large components → use bridge_components (iterative)
# If you have many small fragments → use bridge_components_batch (faster)

# Option A — iterative, one bridge at a time
G_connected = bridge_components(
    G, points,
    target_components=1,       # how many components you want at the end
    max_bridge_distance=10.0,  # tune this: max voxel distance for a bridge
    verbose=True
)

# Option B — batch, bridges all components in one MST pass per iteration  
G_connected = bridge_components_batch(
    G, points,
    target_components=1,
    max_bridge_distance=10.0,
    verbose=True
)

# Then continue your pipeline
cycle_results = SkelToGraph.process_cycles(G_connected, min_cycle_edges=4, min_shared_nodes=2)
G_simplified  = simplify_graph(G_connected, cycle_results)

In [ ]:
SkelToGraph.plot_graph_3d(G_simplified, points, output_file='simplified.html', title='Simplified', edge_width=7)